# GDF Signal Exploration — BCI Competition IV Dataset 2b

**Objective:** Visual comparison between the raw EEG signal and each stage of the processing pipeline.

**Pipeline stages covered:**
1. Raw GDF — as recorded
2. Band-pass filtered (8–30 Hz, mu + beta bands)
3. Epoch segmentation aligned to motor imagery cues
4. ERSP spectrogram (input to the CNN models)

---
**Dataset:** BCI-IV-2b | Channels: C3, Cz, C4 | Fs = 250 Hz  
**File naming:** `B{subject:02d}0{session}T.gdf` (training) / `B{subject:02d}0{session}E.gdf` (evaluation)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import welch, stft as scipy_stft
import mne

mne.set_log_level('WARNING')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})

from config import (
    SFREQ, CHANNELS, FILT_LOW, FILT_HIGH,
    EPOCH_TMIN, EPOCH_TMAX, BASELINE,
    ERSP_FMIN, ERSP_FMAX, STFT_WIN_LEN, STFT_HOP,
    CLASS_NAMES
)
from preprocessing import load_raw, apply_filter, extract_epochs
from ersp import compute_ersp_image

print('Modules loaded.')

## Configuration
Change these variables to explore different subjects and sessions.

In [ ]:
SUBJECT  = 1   # 1–9
SESSION  = 1   # 1–3 (training), 4–5 (evaluation)
CHANNEL  = 'C3'  # 'C3', 'Cz', or 'C4'
WINDOW_S = 15    # seconds to show in time-domain plots

print(f'Subject: S{SUBJECT:02d} | Session: {SESSION} | Channel: {CHANNEL}')

---
## 1. Load and inspect the GDF file

In [ ]:
raw = load_raw(SUBJECT, SESSION)

print(f'File: B{SUBJECT:02d}0{SESSION}{"T" if SESSION <= 3 else "E"}.gdf')
print(f'  Channels:      {raw.ch_names}')
print(f'  Sampling rate: {raw.info["sfreq"]} Hz')
print(f'  Duration:      {raw.n_times / raw.info["sfreq"]:.1f} s  ({raw.n_times} samples)')
print(f'  Data range:    [{raw.get_data().min()*1e6:.1f}, {raw.get_data().max()*1e6:.1f}] µV')

# Events in this file
events, event_id = mne.events_from_annotations(raw, verbose=False)
print(f'\nAnnotations/Events:')
for name, code in event_id.items():
    count = (events[:, 2] == code).sum()
    print(f'  [{code:>4}] {name:<30} → {count} occurrences')

---
## 2. Raw vs. filtered signal

Side-by-side comparison for all 3 channels.  
The 8–30 Hz bandpass retains the **mu (8–13 Hz)** and **beta (14–30 Hz)** bands linked to motor imagery.

In [ ]:
raw_filt = raw.copy()
apply_filter(raw_filt)

data_raw  = raw.get_data()      * 1e6   # V → µV
data_filt = raw_filt.get_data() * 1e6
t = raw.times
mask = t < WINDOW_S

n_ch = len(raw.ch_names)
fig, axes = plt.subplots(n_ch, 2, figsize=(16, 3.5 * n_ch), sharex=True)
if n_ch == 1:
    axes = axes.reshape(1, -1)

for i, ch in enumerate(raw.ch_names):
    # Raw
    ax = axes[i, 0]
    ax.plot(t[mask], data_raw[i, mask], color='#2C7BB6', lw=0.5, alpha=0.85)
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'{ch} — Raw signal')
    ax.axhline(0, color='gray', lw=0.4, ls='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Filtered
    ax = axes[i, 1]
    ax.plot(t[mask], data_filt[i, mask], color='#D7191C', lw=0.5, alpha=0.85)
    ax.set_title(f'{ch} — Filtered ({FILT_LOW}–{FILT_HIGH} Hz)')
    ax.axhline(0, color='gray', lw=0.4, ls='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')

fig.suptitle(
    f'S{SUBJECT:02d} Session {SESSION} — Raw vs. filtered (first {WINDOW_S} s)\n'
    f'Blue = raw | Red = band-pass 8–30 Hz',
    fontweight='bold', fontsize=12
)
plt.tight_layout()
plt.show()

---
## 3. Power spectral density — Raw vs. filtered

The PSD shows the frequency content removed by the filter.  
After filtering, only the 8–30 Hz range (shaded) remains.

In [ ]:
ch_idx = raw.ch_names.index(CHANNEL) if CHANNEL in raw.ch_names else 0
ch_label = raw.ch_names[ch_idx]

freqs_r, psd_r = welch(data_raw[ch_idx]  / 1e6, fs=SFREQ, nperseg=512, noverlap=256)
freqs_f, psd_f = welch(data_filt[ch_idx] / 1e6, fs=SFREQ, nperseg=512, noverlap=256)

psd_r_db = 10 * np.log10(psd_r + 1e-30)
psd_f_db = 10 * np.log10(psd_f + 1e-30)

fig, ax = plt.subplots(figsize=(10, 4))

mask_f = freqs_r <= 60
ax.plot(freqs_r[mask_f], psd_r_db[mask_f], color='#2C7BB6', lw=1.2, label='Raw', alpha=0.9)
ax.plot(freqs_f[mask_f], psd_f_db[mask_f], color='#D7191C', lw=1.2, label='Filtered (8–30 Hz)', alpha=0.9)

ax.axvspan(FILT_LOW, FILT_HIGH, alpha=0.08, color='green', label='Passband (8–30 Hz)')
ax.axvspan(8,  13, alpha=0.12, color='cyan',   label='Mu band (8–13 Hz)')
ax.axvspan(14, 30, alpha=0.10, color='orange', label='Beta band (14–30 Hz)')

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (dB/Hz)')
ax.set_title(f'S{SUBJECT:02d} Session {SESSION} — Power Spectral Density ({ch_label})', fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## 4. Events on the filtered signal

Visualise the motor imagery cue markers overlaid on the EEG.  
Each vertical line marks the start of a motor imagery trial.

In [ ]:
from config import EVENT_LEFT, EVENT_RIGHT, EVENT_LEFT_ONLINE, EVENT_RIGHT_ONLINE

# Identify left and right event codes in this file
left_codes  = {str(EVENT_LEFT), str(EVENT_LEFT_ONLINE),  '769', '781'}
right_codes = {str(EVENT_RIGHT), str(EVENT_RIGHT_ONLINE), '770', '783'}

left_times  = [e[0] / SFREQ for e in events if str(e[2]) in left_codes  or
               any(str(e[2]) == str(v) for k, v in event_id.items() if k in left_codes)]
right_times = [e[0] / SFREQ for e in events if str(e[2]) in right_codes or
               any(str(e[2]) == str(v) for k, v in event_id.items() if k in right_codes)]

# Fallback: use direct event IDs from the file
if not left_times and not right_times:
    target_ids = {}
    for k, v in event_id.items():
        if k in left_codes:
            target_ids['left'] = v
        elif k in right_codes:
            target_ids['right'] = v
    left_times  = [e[0]/SFREQ for e in events if e[2] == target_ids.get('left')]
    right_times = [e[0]/SFREQ for e in events if e[2] == target_ids.get('right')]

# Show first 120 s
view_s = 120
mask_view = t < view_s

fig, ax = plt.subplots(figsize=(16, 3.5))
ax.plot(t[mask_view], data_filt[ch_idx, mask_view], color='#444444', lw=0.5, alpha=0.8)

for lt in left_times:
    if lt < view_s:
        ax.axvline(lt, color='#2C7BB6', lw=1.0, alpha=0.8, label='Left' if lt == left_times[0] else '')
for rt in right_times:
    if rt < view_s:
        ax.axvline(rt, color='#D7191C', lw=1.0, alpha=0.8, label='Right' if rt == right_times[0] else '')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (µV)')
ax.set_title(
    f'S{SUBJECT:02d} Session {SESSION} — Filtered signal with cue markers ({ch_label}, first {view_s} s)\n'
    f'Blue = left imagery | Red = right imagery',
    fontweight='bold'
)
ax.legend(fontsize=8, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Total cue markers visible (first {view_s} s):')
print(f'  Left:  {sum(1 for t_ in left_times  if t_ < view_s)}')
print(f'  Right: {sum(1 for t_ in right_times if t_ < view_s)}')

---
## 5. Epoch comparison — Left vs. Right imagery

Each epoch goes from **-0.5 s** (baseline) to **+4.0 s** (motor imagery window).  
The shaded area shows mean ± 1 SD across all trials of the class.

In [ ]:
epochs = extract_epochs(raw_filt.copy())

print(f'Epochs extracted:')
print(f'  Total:  {len(epochs)}')
print(f'  Left:   {len(epochs["left"])  if "left"  in epochs.event_id else 0}')
print(f'  Right:  {len(epochs["right"]) if "right" in epochs.event_id else 0}')
print(f'  Window: {EPOCH_TMIN} → {EPOCH_TMAX} s  ({epochs.get_data().shape[-1]} samples/epoch)')

# Average ERP per class for all channels
n_ch = len(epochs.ch_names)
fig, axes = plt.subplots(1, n_ch, figsize=(5.5 * n_ch, 4), sharey=True)
if n_ch == 1:
    axes = [axes]

cls_styles = [
    ('left',  '#2C7BB6', 'Left hand imagery'),
    ('right', '#D7191C', 'Right hand imagery'),
]

for ax_i, ch in enumerate(epochs.ch_names):
    ax = axes[ax_i]
    for cls_name, color, label in cls_styles:
        if cls_name not in epochs.event_id:
            continue
        ep = epochs[cls_name].get_data(picks=[ax_i])[:, 0, :] * 1e6
        mean = ep.mean(axis=0)
        std  = ep.std(axis=0)
        ax.plot(epochs.times, mean, color=color, lw=1.5, label=f'{label} (n={len(ep)})')
        ax.fill_between(epochs.times, mean - std, mean + std, color=color, alpha=0.15)

    ax.axvline(0,    color='black', lw=0.9, ls='--', label='Cue onset')
    ax.axvspan(EPOCH_TMIN, 0, alpha=0.06, color='gray', label='Baseline')
    ax.axhline(0, color='gray', lw=0.4, ls=':')
    ax.set_title(f'Channel {ch}')
    ax.set_xlabel('Time relative to onset (s)')
    if ax_i == 0:
        ax.set_ylabel('Mean amplitude (µV) ± SD')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    f'S{SUBJECT:02d} Session {SESSION} — Average epoch per class (mean ± 1 SD)',
    fontweight='bold', fontsize=12
)
plt.tight_layout()
plt.show()

---
## 6. Individual epoch viewer — signal + spectrogram

For a selected trial index, shows the raw epoch waveform alongside its short-time Fourier transform (STFT) spectrogram.  
Change `TRIAL_IDX` to browse different trials.

In [ ]:
TRIAL_IDX = 0   # trial index within each class (0 = first trial)

ch_idx = epochs.ch_names.index(CHANNEL) if CHANNEL in epochs.ch_names else 0
ch_label = epochs.ch_names[ch_idx]
bl_end = int(abs(EPOCH_TMIN) * SFREQ)   # sample index where baseline ends

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

cls_styles = [
    ('left',  '#2C7BB6', 'Left'),
    ('right', '#D7191C', 'Right'),
]

for row, (cls_name, color, cls_display) in enumerate(cls_styles):
    if cls_name not in epochs.event_id:
        continue
    ep_data = epochs[cls_name].get_data()  # (N, C, T)
    if TRIAL_IDX >= len(ep_data):
        print(f'  Trial {TRIAL_IDX} not available for {cls_display} (only {len(ep_data)} trials). Using 0.')
        idx = 0
    else:
        idx = TRIAL_IDX

    signal   = ep_data[idx, ch_idx, :]   # (T,)
    baseline = signal[:bl_end]

    # ── Panel A: time domain ──
    ax_time = fig.add_subplot(gs[row, 0])
    ax_time.plot(epochs.times, signal * 1e6, color=color, lw=0.9)
    ax_time.axvline(0, color='black', lw=0.9, ls='--')
    ax_time.axvspan(EPOCH_TMIN, 0, alpha=0.08, color='gray')
    ax_time.set_title(f'{cls_display} — EEG waveform ({ch_label}, trial {idx})')
    ax_time.set_xlabel('Time (s)')
    ax_time.set_ylabel('Amplitude (µV)')
    ax_time.spines['top'].set_visible(False)
    ax_time.spines['right'].set_visible(False)

    # ── Panel B: STFT spectrogram (raw power) ──
    ax_stft = fig.add_subplot(gs[row, 1])
    freqs_stft, times_stft, Zxx = scipy_stft(
        signal, fs=SFREQ,
        window='hann', nperseg=STFT_WIN_LEN,
        noverlap=STFT_WIN_LEN - STFT_HOP, padded=True
    )
    power_db = 10 * np.log10(np.abs(Zxx) ** 2 + 1e-30)
    f_mask = (freqs_stft >= 4) & (freqs_stft <= 40)
    t_offset = times_stft + EPOCH_TMIN
    im1 = ax_stft.pcolormesh(
        t_offset, freqs_stft[f_mask], power_db[f_mask],
        cmap='RdYlBu_r', shading='auto'
    )
    ax_stft.axvline(0, color='white', lw=1.0, ls='--')
    ax_stft.axhline(FILT_LOW,  color='white', lw=0.6, ls=':', alpha=0.6)
    ax_stft.axhline(FILT_HIGH, color='white', lw=0.6, ls=':', alpha=0.6)
    ax_stft.set_title(f'{cls_display} — STFT (raw power, dB)')
    ax_stft.set_xlabel('Time (s)')
    ax_stft.set_ylabel('Frequency (Hz)')
    plt.colorbar(im1, ax=ax_stft, label='dB')

    # ── Panel C: ERSP image (CNN input) ──
    ax_ersp = fig.add_subplot(gs[row, 2])
    ersp_img = compute_ersp_image(signal, baseline, sfreq=SFREQ)
    im2 = ax_ersp.imshow(
        ersp_img, aspect='auto', origin='lower', cmap='RdYlBu_r',
        extent=[EPOCH_TMIN, EPOCH_TMAX, ERSP_FMIN, ERSP_FMAX],
        vmin=0, vmax=1
    )
    ax_ersp.axvline(0, color='white', lw=1.0, ls='--')
    ax_ersp.axhspan(8,  13, alpha=0.15, color='cyan')
    ax_ersp.axhspan(14, 30, alpha=0.10, color='yellow')
    ax_ersp.set_title(f'{cls_display} — ERSP (CNN input, normalised)')
    ax_ersp.set_xlabel('Time (s)')
    ax_ersp.set_ylabel('Frequency (Hz)')
    plt.colorbar(im2, ax=ax_ersp, label='Norm. ERSP [0–1]')

fig.suptitle(
    f'S{SUBJECT:02d} Session {SESSION} | Channel {ch_label} | Trial {TRIAL_IDX}\n'
    f'Waveform → STFT spectrogram → ERSP image (CNN input)',
    fontweight='bold', fontsize=12
)
plt.show()

---
## 7. Average ERSP image per class — all channels

Average across all trials of each class. This is the signal that the CNN models learn to discriminate.

In [ ]:
cls_styles = [
    ('left',  'Left hand imagery'),
    ('right', 'Right hand imagery'),
]

n_ch = len(epochs.ch_names)
n_cls = sum(1 for c, _ in cls_styles if c in epochs.event_id)

fig, axes = plt.subplots(n_cls, n_ch, figsize=(5.5 * n_ch, 4.5 * n_cls))
if n_cls == 1 or n_ch == 1:
    axes = np.array(axes).reshape(n_cls, n_ch)

row = 0
for cls_name, cls_label in cls_styles:
    if cls_name not in epochs.event_id:
        continue
    ep_data = epochs[cls_name].get_data()  # (N, C, T)
    bl_end_  = int(abs(EPOCH_TMIN) * SFREQ)

    for col, ch in enumerate(epochs.ch_names):
        ersp_stack = []
        for trial in range(len(ep_data)):
            sig = ep_data[trial, col, :]
            bl  = sig[:bl_end_]
            img = compute_ersp_image(sig, bl, sfreq=SFREQ)
            ersp_stack.append(img)

        ersp_mean = np.mean(ersp_stack, axis=0)

        ax = axes[row, col]
        im = ax.imshow(
            ersp_mean, aspect='auto', origin='lower', cmap='RdYlBu_r',
            extent=[EPOCH_TMIN, EPOCH_TMAX, ERSP_FMIN, ERSP_FMAX],
            vmin=0, vmax=1
        )
        ax.axvline(0, color='white', lw=1.0, ls='--')
        ax.axhspan(8,  13, alpha=0.2, color='cyan',   label='Mu'   if col == 0 else '')
        ax.axhspan(14, 30, alpha=0.15, color='yellow', label='Beta' if col == 0 else '')
        ax.set_title(f'{cls_label} — {ch} (n={len(ep_data)})')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Frequency (Hz)' if col == 0 else '')
        plt.colorbar(im, ax=ax, label='Norm. ERSP [0–1]')

    row += 1

fig.suptitle(
    f'S{SUBJECT:02d} Session {SESSION} — Average ERSP per class and channel\n'
    f'(cyan = mu 8–13 Hz | yellow = beta 14–30 Hz)',
    fontweight='bold', fontsize=12
)
plt.tight_layout()
plt.show()

---
## 8. Full pipeline overlay — Raw → Filtered → Epoch (single trial)

Zoomed view of **one trial window** showing each processing stage stacked.  
The gray shaded region is the baseline; the dashed line marks cue onset.

In [ ]:
CLS_SHOW   = 'left'   # 'left' or 'right'
TRIAL_SHOW = 0        # trial index

ch_idx_show = raw.ch_names.index(CHANNEL) if CHANNEL in raw.ch_names else 0

if CLS_SHOW in epochs.event_id and len(epochs[CLS_SHOW]) > TRIAL_SHOW:
    # Get the onset sample from the epochs object
    cls_events = epochs[CLS_SHOW].events
    onset_sample = cls_events[TRIAL_SHOW, 0]  # sample in the continuous recording
    onset_s      = onset_sample / SFREQ

    # Extract the raw window
    t_start = int((onset_s + EPOCH_TMIN) * SFREQ)
    t_end   = int((onset_s + EPOCH_TMAX) * SFREQ)
    t_start = max(t_start, 0)
    t_end   = min(t_end, raw.n_times)

    win_raw   = data_raw[ch_idx_show,  t_start:t_end] 
    win_filt  = data_filt[ch_idx_show, t_start:t_end]
    win_epoch = epochs[CLS_SHOW].get_data()[TRIAL_SHOW, ch_idx_show, :] * 1e6

    n_samples = min(len(win_raw), len(win_filt), len(win_epoch))
    t_win = np.linspace(EPOCH_TMIN, EPOCH_TMAX, n_samples)

    fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)

    for ax, sig, label, color in [
        (axes[0], win_raw[:n_samples],   'Raw EEG',           '#2C7BB6'),
        (axes[1], win_filt[:n_samples],  f'Filtered {FILT_LOW}–{FILT_HIGH} Hz', '#D7191C'),
        (axes[2], win_epoch[:n_samples], 'Epoch (baseline-corrected)', '#1A9641'),
    ]:
        ax.plot(t_win, sig, color=color, lw=0.8)
        ax.axvline(0,         color='black', lw=1.0, ls='--', label='Cue onset')
        ax.axvspan(EPOCH_TMIN, 0, alpha=0.08, color='gray', label='Baseline')
        ax.axhline(0, color='gray', lw=0.4, ls=':')
        ax.set_ylabel('µV')
        ax.set_title(label)
        ax.legend(fontsize=8, loc='upper right')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[-1].set_xlabel('Time relative to cue onset (s)')
    fig.suptitle(
        f'S{SUBJECT:02d} Session {SESSION} | {CHANNEL} | '
        f'{CLS_SHOW.capitalize()} imagery | Trial {TRIAL_SHOW}\n'
        f'Full pipeline: Raw → Filtered → Baseline-corrected epoch',
        fontweight='bold', fontsize=12
    )
    plt.tight_layout()
    plt.show()
else:
    print(f'Class "{CLS_SHOW}" not available or trial {TRIAL_SHOW} out of range.')

---
## Summary

| Stage | What changes |
|---|---|
| **Raw** | Full-bandwidth EEG, includes slow drifts, 50 Hz noise, muscle/ocular artefacts |
| **Filtered (8–30 Hz)** | Retains mu and beta bands only; drifts and high-freq noise removed |
| **Epoch + baseline** | Segmented to trial window; baseline subtraction removes DC offset |
| **ERSP** | Normalised power ratio (dB) relative to baseline; input image to CNN |

To explore other subjects or sessions, change `SUBJECT`, `SESSION`, and `CHANNEL` in the **Configuration** cell at the top.